# Deep Survival Machines (DSM) baseline

DSM (Nagpal et al., 2021) is the closest deep baseline to VaDeSC: each patient's survival curve is a mixture of Weibull distributions, and a neural network gives the mixture weights and parameters from the covariates. Unlike VaDeSC it has no generative model of the covariates, so its mixture components are not clusters of patients.

We use the official code (auton-survival), copied into `src/dsm/` because the package does not install with torch 2 / numpy 2. The wrapper is `run_dsm` in `src/baselines.py`:
- number of mixture components $k \in \{3, 4, 6\}$ and hidden layers $\in \{[50], [100], [100, 100]\}$, chosen by the validation negative log-likelihood,
- training with the official `train_dsm` (ELBO loss, early stopping on the validation loss),
- risk score for the C-index: $1 - S(t^*)$ with $t^*$ the median training event time.

In [1]:
import sys
sys.path.append("..")
from dataclasses import replace

import numpy as np
import pandas as pd

from src.baselines import run_cox, run_dsm, run_rsf
from src.data import load_simulated, load_sksurv, load_support
from src.vadesc import evaluate_vadesc, train_vadesc

QUICK = False
SEEDS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
N, P, K, D_LAT = (2000, 50, 3, 8) if QUICK else (10_000, 1000, 3, 16)  # same as 03_simulated_benchmark_multi
REAL_SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 60 if QUICK else 300
GRID = dict(ks=(3,), layers=((100,),)) if QUICK else {}

## 1. Simulated data

Same data as `03_simulated_benchmark_multi`; the other models are taken from `results/sim_by_seed.csv`. Seed 3 has no saved IBS for those models, so the IBS column uses seeds 0, 1, 2 and 4 for every model.


In [2]:
rows = []
for s in SEEDS:
    d = load_simulated(n=N, p=P, k=K, d_lat=D_LAT, seed=s)
    rows.append({"seed": s, "model": "DSM", **run_dsm(d, seed=s, **GRID)})
    print("seed", s, "done", flush=True)
sim = pd.DataFrame(rows)
prev = pd.read_csv("../results/sim_by_seed.csv")
prev = prev[prev.model.isin(["Cox PH", "RSF", "Weibull AFT", "VaDeSC"]) & prev.seed.isin(SEEDS)]
if QUICK:  # the saved results are for the full-size data
    prev = prev.iloc[:0]
both = pd.concat([prev[["seed", "model", "C-index", "IBS"]], sim], ignore_index=True)
print(sim[["seed", "k", "layers"]].to_string(index=False))
# Seed 3 has no IBS for the saved models (computed before a fix in metrics.py), so IBS is averaged
# over the seeds where every model has one; the C-index uses all seeds.
ibs_seeds = sorted(set(both.seed) - set(both.loc[both.IBS.isna(), "seed"]))
print("C-index seeds:", sorted(set(both.seed)), "| IBS seeds:", ibs_seeds)
models = ["Cox PH", "RSF", "Weibull AFT", "DSM", "VaDeSC"]
pd.concat([both.groupby("model")[["C-index"]].agg(["mean", "std"]),
           both[both.seed.isin(ibs_seeds)].groupby("model")[["IBS"]].agg(["mean", "std"])], axis=1).reindex(models).round(3)

seed 0 done


seed 1 done


seed 2 done


seed 3 done


seed 4 done


 seed  k     layers
    0  3 (100, 100)
    1  4 (100, 100)
    2  3 (100, 100)
    3  6 (100, 100)
    4  6 (100, 100)
C-index seeds: [0, 1, 2, 3, 4] | IBS seeds: [0, 1, 2, 4]


C-index           IBS       
               mean    std   mean    std
model                                   
Cox PH        0.757  0.021  0.167  0.013
RSF           0.743  0.019  0.183  0.008
Weibull AFT   0.755  0.019  0.184  0.011
DSM           0.759  0.015  0.155  0.014
VaDeSC        0.773  0.035  0.156  0.013

## 2. Real datasets

Cox, RSF, DSM and VaDeSC on the same splits (seeds change the split and the initialisation).

- SUPPORT with the paper's preprocessing (includes the post-baseline variables, for comparison with the paper) and baseline-only (no leakage).
- FLChain, GBSG2, WHAS500, with the paper's FLChain config for VaDeSC. WHAS500's `los` is dropped.

In [3]:
def drop_features(d, names):
    keep = [i for i, f in enumerate(d.features) if f not in names]
    return replace(d, X_tr=d.X_tr[:, keep], X_va=d.X_va[:, keep], X_te=d.X_te[:, keep],
                   features=[d.features[i] for i in keep])

DATASETS = {
    "SUPPORT (paper preprocessing)": (lambda s: load_support(seed=s), dict(K=4, d_lat=16, weibull_k=2.0)),
    "SUPPORT (baseline-only)": (lambda s: load_support(drop_post_baseline=True, seed=s), dict(K=4, d_lat=16, weibull_k=2.0)),
    "FLChain": (lambda s: load_sksurv("FLChain", seed=s), dict(K=2, d_lat=4, weibull_k=1.0)),
    "GBSG2": (lambda s: load_sksurv("GBSG2", seed=s), dict(K=2, d_lat=4, weibull_k=1.0)),
    "WHAS500": (lambda s: drop_features(load_sksurv("WHAS500", seed=s), ["los"]), dict(K=2, d_lat=4, weibull_k=1.0)),
}

rows = []
for name, (load, cfg) in DATASETS.items():
    for s in REAL_SEEDS:
        d = load(s)
        rows.append({"dataset": name, "seed": s, "model": "Cox PH", **run_cox(d)})
        rows.append({"dataset": name, "seed": s, "model": "RSF", **run_rsf(d, seed=s)})
        rows.append({"dataset": name, "seed": s, "model": "DSM", **run_dsm(d, seed=s, **GRID)})
        m, _, _ = evaluate_vadesc(train_vadesc(d, epochs=EPOCHS, seed=s, **cfg), d)
        rows.append({"dataset": name, "seed": s, "model": "VaDeSC", "C-index": m["C-index"], "IBS": m["IBS"]})
        print(name, "seed", s, "done", flush=True)
real = pd.DataFrame(rows)
(real.groupby(["dataset", "model"])[["C-index", "IBS"]].agg(["mean", "std"])
     .reindex(list(DATASETS), level=0).reindex(["Cox PH", "RSF", "DSM", "VaDeSC"], level=1).round(3))

SUPPORT (paper preprocessing) seed 0 done


SUPPORT (paper preprocessing) seed 1 done


SUPPORT (paper preprocessing) seed 2 done


SUPPORT (baseline-only) seed 0 done


SUPPORT (baseline-only) seed 1 done


SUPPORT (baseline-only) seed 2 done


... (6 outputs left out)

WHAS500 seed 0 done


WHAS500 seed 1 done


WHAS500 seed 2 done


C-index           IBS       
                                        mean    std   mean    std
dataset                       model                              
SUPPORT (paper preprocessing) Cox PH   0.811  0.024  0.142  0.019
                              RSF      0.804  0.003  0.131  0.017
                              DSM      0.825  0.027  0.131  0.017
                              VaDeSC   0.801  0.041  0.153  0.024
SUPPORT (baseline-only)       Cox PH   0.722  0.007  0.181  0.002
                              RSF      0.717  0.005  0.178  0.001
                              DSM      0.727  0.007  0.178  0.001
                              VaDeSC   0.659  0.013  0.215  0.010
FLChain                       Cox PH   0.795  0.002  0.109  0.001
                              RSF      0.792  0.006  0.109  0.002
                              DSM      0.797  0.005  0.109  0.002
                              VaDeSC   0.799  0.003  0.110  0.002
GBSG2                         Cox PH   0.651  0.023  0.187  0.005
                              RSF      0.673  0.032  0.181  0.005
                              DSM      0.660  0.037  0.183  0.006
                              VaDeSC   0.650  0.020  0.190  0.003
WHAS500                       Cox PH   0.754  0.025  0.169  0.017
                              RSF      0.769  0.018  0.169  0.011
                              DSM      0.752  0.013  0.166  0.004
                              VaDeSC   0.744  0.030  0.185  0.031

In [4]:
# DSM hyperparameters chosen on the validation set
real[real.model == "DSM"].pivot_table(index="dataset", columns="seed", values="k", aggfunc="first")

seed,0,1,2
dataset,,,
FLChain,4.0,4.0,3.0
GBSG2,3.0,4.0,6.0
SUPPORT (baseline-only),6.0,6.0,6.0
SUPPORT (paper preprocessing),6.0,6.0,3.0
WHAS500,6.0,6.0,3.0


## Notes

- The paper tunes DSM with its own grid; ours is smaller (9 settings), so DSM might do slightly better with more tuning.
- DSM gives a full survival curve, so it has an IBS; its mixture components are not used as clusters here.